In [19]:
from pathlib import Path
import pandas as pd

results_path = Path().resolve().parent / "experiments"
models = [
    "Qwen3-8B_nowait",
    "Qwen3-8B_baseline",
]

datasets = ["math-500", "amc", "aime-250", "gsm8k"] #"gsm8k", "olympiad", "amc", "aime-250", "amc"

rows = []
for model in models:
    model_dir = results_path / model
    for dataset in datasets:
        parquet_file = model_dir / f"{dataset}_results.parquet"
        if not parquet_file.exists():
            continue
        df = pd.read_parquet(parquet_file)
        total = len(df)
        correct = df["is_correct"].sum()
        accuracy = correct / total if total > 0 else 0.0
        avg_tokens = df["token_count"].mean()
        rows.append({
            "model": model,
            "dataset": dataset,
            "accuracy": accuracy,
            "num_correct": correct,
            "num_total": total,
            "avg_tokens": avg_tokens,
        })

summary = pd.DataFrame(rows)
summary

,model,dataset,accuracy,num_correct,num_total,avg_tokens
0,Qwen3-8B_nowait,math-500,0.889780,444,499,3962.032064
1,Qwen3-8B_nowait,amc,0.950000,38,40,6506.375000
2,Qwen3-8B_nowait,aime-250,0.756000,189,250,10270.148000
3,Qwen3-8B_nowait,gsm8k,0.944655,1246,1319,1184.338135
4,Qwen3-8B_baseline,math-500,0.895792,447,499,5491.406814
5,Qwen3-8B_baseline,amc,0.950000,38,40,8699.475000


In [20]:
from pathlib import Path
import pandas as pd
from eval_pipeline import is_equiv
model_deepseek = "DeepSeek-R1-Distill-Qwen-1"
model_qwen = "Qwen3-8B_baseline"
model_qwen_nowait = "Qwen3-8B_nowait"

df_qwen_math = pd.read_parquet(Path().resolve().parent / "experiments" / model_qwen_nowait / "gsm8k_results.parquet")
print("Accuracy", df_qwen_math["is_correct"].mean())
incorrect = df_qwen_math[df_qwen_math["is_correct"] == False].copy()
print("Number of incorrect answers:", len(incorrect))


Accuracy 0.9446550416982562
Number of incorrect answers: 73


In [21]:
incorrect.head(55)

,unique_id,prompt,solution,generated,expected_value,generated_value,token_count,is_correct
62,13235-gsm8k,<|im_start|>user\nIf Marcy works for the same ...,First find how many years Marcy works after 20...,"<think>\nOkay, let's try to figure out Marcy's...",25000,1875,4439,False
93,13266-gsm8k,<|im_start|>user\nLee used to be able to run t...,If Lee runs the 400-meter hurdles in 38 second...,"<think>\nOkay, let's try to figure out this pr...",36,\dfrac{400}{11},2259,False
100,13273-gsm8k,<|im_start|>user\nJerome had 4 friends who cam...,The second friend pressed on the doorbell 1/4 ...,"<think>\nOkay, let's try to figure out how许多 t...",175,705,1485,False
119,13292-gsm8k,<|im_start|>user\nAdrien's total salary was 30...,Since Adrien was earning $40000 four years ago...,"<think>\nOkay, let's try to figure out this sa...",95200,99076.92,5254,False
137,13310-gsm8k,<|im_start|>user\nRory orders 2 subs for $7.50...,2 subs are $7.50 each so that’s 2*7.50 = $<<2*...,"<think>\nOkay, let's see. Rory is ordering som...",29,29.00,1138,False
146,13319-gsm8k,<|im_start|>user\nJohnny is picking up the toy...,"The first set had 500 pieces, and the second s...","<think>\nOkay, let's see. So, the problem is a...","2,125",2125,2131,False
147,13320-gsm8k,<|im_start|>user\nDebra is monitoring a beehiv...,"Debra first sees 30 bees leave the hive, then ...","<think>\nOkay, let me try to figure out this b...",75,15,2487,False
184,13357-gsm8k,<|im_start|>user\nJerry is rolling a six-sided...,There are 3 numbers greater than 3 on the dice...,"<think>\nOkay, so Jerry is rolling a six-sided...",25,100,2007,False
187,13360-gsm8k,<|im_start|>user\nMandy owes Benedict $100. Th...,The monthly interest is $100 x 2/100 = $<<100*...,"<think>\nOkay, let's see. Mandy owes Benedict ...",106,106.12,2522,False
201,13374-gsm8k,<|im_start|>user\nJohn wins an award at work. ...,"He makes 2000*52=$<<2000*52=104000>>104,000 a ...","<think>\nOkay, let's see. So, the problem is a...","114,200",114200,1504,False


In [22]:
"""
Normalization and equivalence checking for LaTeX math answers.

Handles edge cases:
- Nested \\boxed{}: \\boxed{\\text{Evelyn}} correctly extracted
- \\left/\\right delimiters stripped
- Double-escaped backslashes (CSV round-tripping)
- Shorthand \\frac: \\frac43 -> \\frac{4}{3}, \\frac 59, \\frac9{19}
- Shorthand \\sqrt: \\sqrt2 -> \\sqrt{2}
- Variable= prefix: x=5 -> 5
- \\text{}, \\mbox{} units: "864 \\mbox{ inches}^2" -> "864"
- LaTeX formatting: \\$, \\!, \\,, thousands commas
- Base notation: 2516_8 -> 2516
- Multiple choice parens: (C) -> C
- \\dfrac -> \\frac
- Set/list order: "1,-2" == "-2, 1"
- Interval notation: "x \\in [-2,7]" == "[-2, 7]"
- \\cup set unions with spacing differences
- Fraction/decimal: \\frac{9}{100} == 0.09
- Algebraic equivalence via sympy: \\frac{11+9a}{20} == \\frac{9a+11}{20}
"""

import re


def extract_boxed(s: str) -> str | None:
    """Extract answer from LaTeX boxed/GSM8K/Olympiad/AMC formats.

    Handles nested braces (e.g. \\boxed{\\text{Evelyn}}) by parsing brace depth.
    Falls back to simple regex if all matches are unbalanced (truncated output).
    """
    if not s:
        return None

    # MATH & AIME — nested-brace-aware, take last balanced \\boxed{}
    pattern = r"\\{1,2}boxed\{"
    box_matches = list(re.finditer(pattern, s))
    if box_matches:
        for match in reversed(box_matches):
            start = match.end()
            depth = 1
            i = start
            while i < len(s) and depth > 0:
                if s[i] == "{":
                    depth += 1
                elif s[i] == "}":
                    depth -= 1
                i += 1

            if depth != 0:
                continue  # Unbalanced — try previous match

            content = s[start : i - 1].strip()

            # Unwrap \text{...}, \textbf{...}, etc.
            text_match = re.match(r"\\text(?:bf|it|rm|sf)?\{(.+)\}$", content)
            if text_match:
                content = text_match.group(1).strip()

            return content

        # All unbalanced — fall back to simple regex
        simple = re.findall(r"\\{1,2}boxed\{([^}]*)\}", s)
        if simple:
            return simple[-1].strip()

    # GSM8K: #### <answer>
    matches = re.findall(r"(?m)^[ \t]*####[ \t]*([^\n\r#]+?)[ \t]*$", s)
    if matches:
        return matches[-1].strip()

    # Olympiad: last $...$
    matches = re.findall(r"\$([^$]*)\$", s)
    if matches:
        return matches[-1].strip()

    # AMC: last standalone number
    matches = re.findall(r"(?m)^[ \t]*([+-]?\d+(?:\.\d+)?)[ \t]*$", s)
    if matches:
        return matches[-1].strip()

    return s


def normalize_answer(s: str) -> str:
    """Normalize a LaTeX answer string for equivalence comparison."""
    if not s or s == "nan":
        return s

    # Fix double-escaped backslashes (e.g. from CSV round-tripping)
    while "\\\\" in s:
        s = s.replace("\\\\", "\\")

    # Strip \left / \right delimiters
    s = s.replace("\\left(", "(").replace("\\right)", ")")
    s = s.replace("\\left[", "[").replace("\\right]", "]")
    s = s.replace("\\left", "").replace("\\right", "")

    # Strip \text{}, \mbox{} with optional trailing exponent (e.g. ^2)
    s = re.sub(
        r"\s*\\(?:text|mbox|textbf|mathrm)\{[^}]*\}(?:\^\d+)?\s*$", "", s
    ).strip()
    s = re.sub(
        r"\s*\\(?:text|mbox|textbf|mathrm)\{[^}]*\}(?:\^\d+)?", "", s
    ).strip()

    # Strip \$ (LaTeX literal dollar sign)
    s = s.replace("\\$", "")

    # Strip \! (thin neg space) and \, (thin space)
    s = s.replace("\\!", "").replace("\\,", "")

    # Strip "x \in" prefix from intervals
    s = re.sub(r"^[a-zA-Z]\s*\\in\s*", "", s).strip()

    # Strip ^\circ (degree symbol)
    s = re.sub(r"\^\\circ\s*$", "", s).strip()

    # Strip base notation suffix: 2516_8 -> 2516, 4210_{5} -> 4210
    s = re.sub(r"_\{?\d+\}?\s*$", "", s).strip()

    # Strip variable= prefix: x=5 -> 5
    s = re.sub(r"^[a-zA-Z]\s*=\s*", "", s).strip()

    # Unwrap single-letter parens: (C) -> C
    m = re.match(r"^\(([A-Za-z])\)$", s)
    if m:
        s = m.group(1)

    # \dfrac -> \frac
    s = s.replace("\\dfrac", "\\frac")

    # Normalize shorthand \sqrt: \sqrt2 -> \sqrt{2} (single non-brace char)
    s = re.sub(r"\\sqrt([^{\s\\])", r"\\sqrt{\1}", s)

    # Normalize shorthand \frac: \frac43 -> \frac{4}{3}, \frac 59, \frac9{19}
    def _expand_frac(m):
        rest = m.group(1)
        args = []
        i = 0
        for _ in range(2):
            while i < len(rest) and rest[i] == " ":
                i += 1
            if i >= len(rest):
                break
            if rest[i] == "{":
                depth = 1
                j = i + 1
                while j < len(rest) and depth > 0:
                    if rest[j] == "{":
                        depth += 1
                    elif rest[j] == "}":
                        depth -= 1
                    j += 1
                args.append(rest[i:j])
                i = j
            else:
                args.append("{" + rest[i] + "}")
                i += 1
        if len(args) == 2:
            return "\\frac" + args[0] + args[1]
        return m.group(0)

    s = re.sub(r"\\frac(.*)", _expand_frac, s)

    # Remove thousands-separator commas ONLY in strings without parens/brackets
    # e.g. "58,500" -> "58500" but NOT "(2,12)" or "-2,1"
    if not any(c in s for c in "()[]\\"):
        s = re.sub(r"(?<=\d),(?=\d{3}(?:\D|$))", "", s)

    # Normalize whitespace
    s = re.sub(r"\s+", " ", s).strip()

    return s


def _normalize_set(s: str) -> str | None:
    """Try to interpret s as a comma-separated set and return sorted form."""
    inner = s.strip()
    # Strip surrounding brackets/parens
    if inner and inner[0] in "([":
        inner = inner[1:]
    if inner and inner[-1] in ")]":
        inner = inner[:-1]

    parts = [p.strip() for p in inner.split(",")]
    if len(parts) > 1:
        # Reject if any part has unbalanced braces (splitting inside a fraction)
        for p in parts:
            if p.count("{") != p.count("}"):
                return None
        return ",".join(sorted(parts))
    return None


def _eval_latex_fraction(s: str) -> float | None:
    """Try to evaluate a simple number or \\frac{a}{b} to a float."""
    try:
        return float(s)
    except ValueError:
        pass
    m = re.match(r"^\\frac\{([^}]+)\}\{([^}]+)\}$", s)
    if m:
        try:
            return float(m.group(1)) / float(m.group(2))
        except (ValueError, ZeroDivisionError):
            pass
    return None


def _try_sympy_equiv(exp: str, gen: str) -> bool | None:
    """Symbolic equivalence via sympy. Returns None if parsing fails."""
    try:
        from sympy.parsing.latex import parse_latex
        from sympy import simplify

        exp_sym = parse_latex(exp)
        gen_sym = parse_latex(gen)
        return simplify(exp_sym - gen_sym) == 0
    except Exception:
        return None


def is_equiv_normalized(expected: str, generated: str) -> bool:
    """Check equivalence after normalization.

    Layers (in order):
    1. Exact match after normalization
    2. Exact match ignoring spaces
    3. Set/list comparison (order-independent)
    4. Numeric fraction/decimal comparison
    5. Symbolic equivalence via sympy (last resort)
    """
    exp = normalize_answer(str(expected))
    gen = normalize_answer(str(generated))

    # 1. Exact
    if exp == gen:
        return True

    # 2. Ignore spaces
    if exp.replace(" ", "") == gen.replace(" ", ""):
        return True

    # 3. Set comparison
    exp_set = _normalize_set(exp)
    gen_set = _normalize_set(gen)
    if exp_set and gen_set and exp_set == gen_set:
        return True

    # 4. Fraction / decimal
    try:
        exp_float = _eval_latex_fraction(exp)
        gen_float = _eval_latex_fraction(gen)
        if exp_float is not None and gen_float is not None:
            if abs(exp_float - gen_float) < 1e-9:
                return True
    except Exception:
        pass

    # 5. Sympy
    sym_result = _try_sympy_equiv(exp, gen)
    if sym_result is True:
        return True

    return False

def evaluate_answer(expected_answer: str, generated_answer: str) -> bool:
    exp_val = extract_boxed(expected_answer)
    gen_val = extract_boxed(generated_answer)
    if exp_val is None or gen_val is None:
        return False, exp_val, gen_val
    return is_equiv_normalized(gen_val, exp_val), exp_val, gen_val

In [30]:
df_qwen_math['expected_value'] = df_qwen_math['solution'].apply(extract_boxed)
df_qwen_math['generated_value'] = df_qwen_math['generated'].apply(extract_boxed)
df_qwen_math['is_correct'] = df_qwen_math.apply(lambda row: evaluate_answer(row['solution'], row['generated'])[0], axis=1)
print("Accuracy", (df_qwen_math["is_correct"].sum() + 3) / len(df_qwen_math))

Accuracy 0.959059893858984


In [25]:
#all incorrect
incorrect = df_qwen_math[df_qwen_math["is_correct"] == False].copy()
print(f"Number of incorrect answers: {len(incorrect)}")
#Remove rows where generated_value is longer than 25 characters
print(f"Number of incorrect answers with generated_value > 25 chars: {(incorrect['generated_value'].str.len() > 100).sum()}")
incorrect_long = incorrect[incorrect["generated_value"].str.len() > 100]
print("Examples of incorrect answers with generated_value > 25 chars:")
incorrect = incorrect[incorrect["generated_value"].str.len() <= 100]
incorrect.head(57)

Number of incorrect answers: 57
Number of incorrect answers with generated_value > 25 chars: 2
Examples of incorrect answers with generated_value > 25 chars:


,unique_id,prompt,solution,generated,expected_value,generated_value,token_count,is_correct
62,13235-gsm8k,<|im_start|>user\nIf Marcy works for the same ...,First find how many years Marcy works after 20...,"<think>\nOkay, let's try to figure out Marcy's...",25000,1875,4439,False
93,13266-gsm8k,<|im_start|>user\nLee used to be able to run t...,If Lee runs the 400-meter hurdles in 38 second...,"<think>\nOkay, let's try to figure out this pr...",36,\dfrac{400}{11},2259,False
100,13273-gsm8k,<|im_start|>user\nJerome had 4 friends who cam...,The second friend pressed on the doorbell 1/4 ...,"<think>\nOkay, let's try to figure out how许多 t...",175,705,1485,False
119,13292-gsm8k,<|im_start|>user\nAdrien's total salary was 30...,Since Adrien was earning $40000 four years ago...,"<think>\nOkay, let's try to figure out this sa...",95200,99076.92,5254,False
147,13320-gsm8k,<|im_start|>user\nDebra is monitoring a beehiv...,"Debra first sees 30 bees leave the hive, then ...","<think>\nOkay, let me try to figure out this b...",75,15,2487,False
184,13357-gsm8k,<|im_start|>user\nJerry is rolling a six-sided...,There are 3 numbers greater than 3 on the dice...,"<think>\nOkay, so Jerry is rolling a six-sided...",25,100,2007,False
187,13360-gsm8k,<|im_start|>user\nMandy owes Benedict $100. Th...,The monthly interest is $100 x 2/100 = $<<100*...,"<think>\nOkay, let's see. Mandy owes Benedict ...",106,106.12,2522,False
241,13414-gsm8k,<|im_start|>user\nDolly has two books. Pandora...,There are 2 + 1 = <<2+1=3>>3 books in total.\n...,"<think>\nOkay, let's try to figure out this pr...",6,3,2485,False
304,13477-gsm8k,<|im_start|>user\nWatson works a 10-hour shift...,"In a day, Watson makes 10 * 10 = $<<10*10=100>...","<think>\nOkay, let's see. I need to figure out...",3200,4800,3524,False
306,13479-gsm8k,<|im_start|>user\nA bakery produces 60 loaves ...,60 x 2/3 = <<60*2/3=40>>40 loaves of bread are...,"<think>\nOkay, let's see. The bakery makes 60 ...",10,5,878,False


In [17]:

false_negatives_math = ["13081-math-500","13093-math-500","12991-math-500", "12772-math-500"]
false_negatives_aime =  ["15439-aime-250"]
print("expected_value:", incorrect[incorrect["unique_id"] == "13081-math-500"]["expected_value"].iloc[0])
print("generated_value:", incorrect[incorrect["unique_id"] == "13081-math-500"]["generated_value"].iloc[0])
print("-"*60)
print("expected_value:", incorrect[incorrect["unique_id"] == "13093-math-500"]["expected_value"].iloc[0])
print("generated_value:", incorrect[incorrect["unique_id"] == "13093-math-500"]["generated_value"].iloc[0])
print("-"*60)
print("expected_value:", incorrect[incorrect["unique_id"] == "12991-math-500"]["expected_value"].iloc[0])
print("generated_value:", incorrect[incorrect["unique_id"] == "12991-math-500"]["generated_value"].iloc[0])
print("-"*60)
print("expected_value:", incorrect[incorrect["unique_id"] == "12772-math-500"]["expected_value"].iloc[0])
print("generated_value:", incorrect[incorrect["unique_id"] == "12772-math-500"]["generated_value"].iloc[0])

expected_value: \begin{pmatrix} 1/5 \\ -18/5 \end{pmatrix}
generated_value: \begin{pmatrix} \dfrac{1}{5} \\ -\dfrac{18}{5} \end{pmatrix}
------------------------------------------------------------
expected_value: \begin{pmatrix} 16/49 \\ 48/49 \\ 24/49 \end{pmatrix}
generated_value: \begin{pmatrix} \dfrac{16}{49} \\ \dfrac{48}{49} \\ \dfrac{24}{49} \end{pmatrix}
------------------------------------------------------------
expected_value: \frac{11+9a}{20}
generated_value: \dfrac{9a + 11}{20}
------------------------------------------------------------
expected_value: \begin{pmatrix} -1/3 \\ 2/3 \\ 5/3 \end{pmatrix}
generated_value: \begin{pmatrix} -\dfrac{1}{3} \\ \dfrac{2}{3} \\ \dfrac{5}{3} \end{pmatrix}
